In [ ]:
import requests
import json

# --- Configuration ---
TOKEN = "YOUR_AUTH_TOKEN"  # Replace with your actual authorization token
TENANT_ID = "c8544681-de6c-4b5c-8705-3df2ce53a2fc" # Replace with your actual tenant_id

llm_ida_headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'Authorization': 'Bearer ' + TOKEN
}

In [ ]:
# --- Knowledge Base Ingestion Functions ---

def create_knowledge_base(kb_name, description, embedding_model="text-embedding-3-large-1", chunk_size=400, inference_model="gpt-4o-2024-05-13"):
    """
    Creates a new knowledge base.
    Configure embedding_model and chunk_size here.
    """
    create_knowledge_base_url = f"https://cs.prod.aws.jpmchase.net/qanda/ingestion/v1/knowledge-base?tenant_id={TENANT_ID}"

    payload = {
        "KnowledgeBaseName": kb_name,
        "EmbeddingSettings": {
            "embedding_model": embedding_model, # Recommend text-embedding-3-large-1
            "chunk_size": chunk_size # Recommended 200-400
        },
        "InferenceSettings": {
            "inference_model": inference_model,
            "max_tokens": 500,
            "search_size": 20,
            "knn_num_neighbors": 100,
            "temperature": 0,
            "assistant_msg": "",
            "hallucination_detection_enabled": False,
            "hallucination_detection_model": "",
            "faq_threshold": 0.95
        },
        "KnowledgeBaseDescription": description
    }

    response = requests.post(create_knowledge_base_url, json=payload, headers=llm_ida_headers)
    if response.status_code == 201:
        print("Knowledge Base created successfully:")
        kb_info = response.json()
        print(json.dumps(kb_info, indent=4))
        return kb_info['knowledgeBaseId']
    else:
        print(f"Failed to create Knowledge Base: {response.status_code} - {response.text}")
        return None

def upload_document_to_kb(knowledge_base_id, file_path, document_type='text/plain', metadata={}):
    """
    Uploads a document to the specified knowledge base.
    The KB API handles chunking and embedding generation.
    You'll need to extend this to pass custom metadata as needed by your API.
    """
    upload_document_url = f"https://cs.prod.aws.jpmchase.net/qanda/ingestion/v1/knowledge-base/upload-document?tenant_id={TENANT_ID}&knowledge_base_id={knowledge_base_id}"

    # You might need to add metadata to the 'doc' tuple or in the query parameters
    # The current API seems to directly embed the document without explicit metadata payload per file.
    # If custom metadata is expected by the KB API for filtering, it needs to be sent either
    # as part of the filename (e.g., 'filename_endpoint_errorcode.txt') and parsed by the API,
    # or as a separate form field, or through a different API endpoint for metadata update.
    # Assuming the API automatically extracts some metadata or uses filename prefix as in image_4d5f21.png
    # For sophisticated metadata, you might need a custom wrapper around this API or a post-upload metadata update API if available.

    files = {
        'doc': (file_path.split('/')[-1], open(file_path, 'rb'), document_type) # file_path.split('/')[-1] extracts file name
    }

    response = requests.post(upload_document_url, headers=llm_ida_headers, files=files)
    if response.status_code == 201:
        print(f"Document '{file_path}' uploaded successfully.")
        return response.json()
    else:
        print(f"Failed to upload document '{file_path}': {response.status_code} - {response.text}")
        return None

# --- Example Ingestion Workflow ---
if __name__ == "__main__":
    # 1. Create a Knowledge Base (if not already created)
    # Be sure to set the correct embedding_model (e.g., text-embedding-3-large-1)
    # and chunk_size (e.g., 200-400) for API log data.
    KB_NAME = "APIPerformanceKB"
    KB_DESCRIPTION = "Knowledge Base for API performance logs, incidents, and documentation."
    knowledge_base_id = create_knowledge_base(KB_NAME, KB_DESCRIPTION,
                                              embedding_model="text-embedding-3-large-1",
                                              chunk_size=300) # Adjust chunk_size as per best practices

    if knowledge_base_id:
        # 2. Prepare your documents (log files, incident reports, runbooks)
        # For structured logs, you'd have a processing step here to
        # format them and extract metadata before writing to temp files or
        # streaming to the API.

        # Example: Simulating log file processing and upload
        # In a real scenario, you'd iterate through your log directories,
        # parse files, extract relevant data points, and then upload.
        
        # Dummy Log File Content
        log_content_1 = """
        {"timestamp": "2025-07-17T10:00:00Z", "service": "auth-service", "endpoint": "/login", "status": 500, "error": "Database connection timed out", "request_id": "req-123", "log_level": "ERROR"}
        {"timestamp": "2025-07-17T10:01:00Z", "service": "order-service", "endpoint": "/create-order", "status": 200, "latency_ms": 150, "log_level": "INFO"}
        {"timestamp": "2025-07-17T10:02:00Z", "service": "payment-service", "endpoint": "/process", "status": 400, "error": "Invalid payment method", "log_level": "WARN"}
        """
        log_content_2 = """
        {"timestamp": "2025-07-18T09:30:00Z", "service": "auth-service", "endpoint": "/verify-token", "status": 503, "error": "Service unavailable", "request_id": "req-456", "log_level": "ERROR"}
        {"timestamp": "2025-07-18T09:31:00Z", "service": "user-profile-service", "endpoint": "/get-profile", "status": 500, "error": "NullPointerException at com.example.User.getProfile", "log_level": "ERROR"}
        """

        # Save to dummy files for demonstration
        with open("dummy_api_log_1.txt", "w") as f:
            f.write(log_content_1)
        with open("dummy_api_log_2.txt", "w") as f:
            f.write(log_content_2)
            
        # Example Incident Report
        incident_report_content = """
        **Incident Report: Payment Service Outage - 2025-07-16**
        **Incident ID:** INC-2025-07-16-001
        **Service Affected:** Payment Gateway Service
        **Impact:** All payment transactions failed for 30 minutes.
        **Root Cause:** A misconfigured database connection pool led to connection exhaustion and subsequent 5xx errors.
        **Resolution:** Rolled back configuration, scaled up database instances.
        **Lessons Learned:** Implement stricter CI/CD checks for database configurations.
        """
        with open("incident_report_payment_service.txt", "w") as f:
            f.write(incident_report_content)

        # Upload the documents
        upload_document_to_kb(knowledge_base_id, "dummy_api_log_1.txt")
        upload_document_to_kb(knowledge_base_id, "dummy_api_log_2.txt")
        upload_document_to_kb(knowledge_base_id, "incident_report_payment_service.txt")

        # In a production setup, you would automate this:
        # - Monitor log directories for new files.
        # - Use a log parsing library (e.g., `json`, `re`, `loguru`) to extract structured data.
        # - For each meaningful log entry or incident, create a "document" (can be a string or temp file)
        #   and associate relevant metadata.
        # - Call `upload_document_to_kb` for each.

In [ ]:
# --- Chatbot Backend Service Functions ---

def invoke_knowledge_base_qa(knowledge_base_id, user_message, conversation_id=None):
    """
    Invokes the Knowledge Base QA API to get an answer using RAG.
    This API handles the retrieval and passes context to the LLM internally.
    """
    qa_url = f"https://cs.prod.aws.jpmchase.net/qanda/inference/v1/qa?tenant_id={TENANT_ID}&knowledge_base_id={knowledge_base_id}"

    payload = {
        "RequestId": "unique_request_id_here", # Generate a unique ID for each request
        "Prompt": [
            {
                "UserMsg": user_message
            }
        ]
    }
    if conversation_id:
        payload["ConversationId"] = conversation_id # Use ConversationId for multi-turn chat

    response = requests.post(qa_url, json=payload, headers=llm_ida_headers)

    if response.status_code == 200:
        qa_response = response.json()
        # The exact structure of the Q&A response needs to be checked.
        # Assuming it returns a direct answer or a structure containing the answer.
        # The LLM API example had "ResponseFromAssistant".
        # You'll likely find the answer within `qa_response.get("Answer")` or similar.
        print("QA API Response:")
        print(json.dumps(qa_response, indent=4))
        # Extract the actual answer from the response
        # This part might need adjustment based on the actual Q&A API response format
        return qa_response.get("Answer", "No answer found.") # Example extraction
    else:
        print(f"QA API Request failed: {response.status_code} - {response.text}")
        return "Sorry, I could not retrieve an answer at this time."

# --- Example Chatbot Interaction ---
if __name__ == "__main__":
    # Assume knowledge_base_id from ingestion step is available
    # knowledge_base_id = "your_actual_kb_id_here" # Get this after creating the KB

    # To demonstrate, let's use a dummy ID if KB creation was skipped in this run
    # In a real app, this would be retrieved from a config or a KB lookup.
    dummy_knowledge_base_id = "your_known_knowledge_base_id" # Replace with a valid KB ID you've created

    print("\n--- Chatbot Interaction Simulation ---")

    # First query
    user_q1 = "What caused the 500 errors for auth-service on 2025-07-17?"
    print(f"\nUser: {user_q1}")
    answer1 = invoke_knowledge_base_qa(dummy_knowledge_base_id, user_q1)
    print(f"Chatbot: {answer1}")

    # Subsequent query in the same conversation (if supported by the API's ConversationId)
    # The Knowledge Base QA API expects ConversationId in the payload
    # For now, we'll use a new conversation ID for each query unless explicitly managed.
    # The LLM API also shows a separate `/invoke/followup` endpoint for multi-turn.
    # You might need to coordinate between the QANDA API and LLM API for complex multi-turn chats.
    # For simplicity, if the QANDA API handles the RAG and LLM call, stick to it.
    
    user_q2 = "Tell me about the incident INC-2025-07-16-001."
    print(f"\nUser: {user_q2}")
    answer2 = invoke_knowledge_base_qa(dummy_knowledge_base_id, user_q2)
    print(f"Chatbot: {answer2}")

    user_q3 = "What should I do if I get a 400 error on /process endpoint?"
    print(f"\nUser: {user_q3}")
    answer3 = invoke_knowledge_base_qa(dummy_knowledge_base_id, user_q3)
    print(f"Chatbot: {answer3}")